In [4]:
import matplotlib.pyplot as plt
import struct
import pandas as pd

def parse_qbb_trace(file_path):
    """
    Parses the binary QBB trace file (TraceFormat struct, 56 bytes each).

    This version matches the exact C++ struct definition and field layout.
    It decodes the union contents based on l3Prot:
      0x6   = TCP   -> data
      0x11  = UDP   -> data
      0xFC/0xFD = ACK
      0xFE  = PFC
      0xFF  = CNP
      else  = qp (default)
    """

    # --- Base struct before the union (12 fields) ---
    base_fmt = "<QHBBIIIHBBBB"
    base_size = struct.calcsize(base_fmt)

    # --- Union structs (from TraceFormat union) ---
    fmt_data = "<HHIQHH"      # sport, dport, seq, ts, pg, payload
    fmt_ack  = "<HHHHIQ"      # sport, dport, flags, pg, seq, ts
    fmt_pfc  = "<IIb3x"       # time, qlen, qIndex (+3 padding to 12 bytes)
    fmt_cnp  = "<HBBHH"       # fid, qIndex, ecnBits, qfb, total
    fmt_qp   = "<HH"          # sport, dport

    rec_size = 56  # confirmed from debugger
    rows = []

    # --- Read file into memory ---
    try:
        with open(file_path, "rb") as f:
            data = f.read()
    except FileNotFoundError:
        print(f"Error: file not found: {file_path}")
        return pd.DataFrame()

    n = len(data)
    if n < rec_size:
        print("Trace file too small — no complete records.")
        return pd.DataFrame()

    off = 0
    last_time = -1

    while off + rec_size <= n:
        # Unpack header (base)
        try:
            header = struct.unpack_from(base_fmt, data, off)
        except struct.error:
            break

        (time_ns, node, intf, qidx, qlen,
         sip, dip, size, l3Prot, event, ecn, nodeType) = header

        union_bytes = data[off + base_size : off + rec_size]

        row = {
            "time": time_ns / 1e9,
            "node": node,
            "intf": intf,
            "qidx": qidx,
            "qlen": qlen,
            "sip": sip,
            "dip": dip,
            "size": size,
            "l3Prot": l3Prot,
            "event": event,
            "ecn": ecn,
            "nodeType": nodeType,
            "sport": None,
            "dport": None,
            "seq": None,
            "ts": None,
            "pg": None,
            "payload": None,
            "ProtType": None,
        }

        # --- Decode union based on l3Prot ---
        try:
            if l3Prot in (0x6, 0x11):  # TCP/UDP data
                (sport, dport, seq, ts, pg, payload) = struct.unpack(fmt_data, union_bytes[:struct.calcsize(fmt_data)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "seq": seq,
                    "ts": ts,
                    "pg": pg,
                    "payload": payload,
                    "ProtType": "TCP" if l3Prot == 0x6 else "UDP",
                })
            elif l3Prot in (0xFC, 0xFD):  # ACK
                (sport, dport, flags, pg, seq, ts) = struct.unpack(fmt_ack, union_bytes[:struct.calcsize(fmt_ack)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "seq": seq,
                    "ts": ts,
                    "pg": pg,
                    "ProtType": "ACK",
                })
            elif l3Prot == 0xFE:  # PFC
                (pfc_time, pfc_qlen, qIndex) = struct.unpack(fmt_pfc, union_bytes[:struct.calcsize(fmt_pfc)])
                row.update({
                    "pfc_time": pfc_time,
                    "pfc_qlen": pfc_qlen,
                    "pfc_qIndex": qIndex,
                    "ProtType": "PFC",
                })
            elif l3Prot == 0xFF:  # CNP
                (fid, qIndex, ecnBits, qfb, total) = struct.unpack(fmt_cnp, union_bytes[:struct.calcsize(fmt_cnp)])
                row.update({
                    "cnp_fid": fid,
                    "cnp_qIndex": qIndex,
                    "cnp_ecnBits": ecnBits,
                    "cnp_qfb": qfb,
                    "cnp_total": total,
                    "ProtType": "CNP",
                })
            else:  # default qp
                (sport, dport) = struct.unpack(fmt_qp, union_bytes[:struct.calcsize(fmt_qp)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "ProtType": "QP",
                })
        except struct.error:
            # corrupted or incomplete record
            pass

        rows.append(row)

        if time_ns < last_time:
            # likely misaligned data — stop
            break
        last_time = time_ns
        off += rec_size

    return pd.DataFrame(rows)

def calculate_throughput(df, interval):
    """
    Calculates throughput in packets over a specified time interval.
    """
    if df.empty:
        return pd.Series()
        
    # Set time as index
    df = df.set_index(pd.to_datetime(df['time'], unit='s'))
    
    # Resample data into time bins and sum the packet coubnt
    throughput = df['size'].resample(f'{interval}s').count()
    
    throughput_gbps = (throughput) / (interval * 1e9)
    
    return throughput_gbps

In [5]:
import plotly.graph_objects as go
import pandas as pd

def plot(df, interval):
    # === Configuration ===
    group_by_node = True      # Toggle: True = per-node plots, False = per-flow plots
    
    # === Build mapping from sip -> node (node is the shorter sender id) ===
    sip_node_map = (
        df[['sip', 'node']]
        .drop_duplicates(subset='sip')
        .set_index('sip')['node']
        .to_dict()
    )

    # === Create short-name columns ===
    df['sip_short'] = df['sip'].map(sip_node_map).fillna(df['sip']).astype(str)
    df['dip_short'] = df['dip'].map(sip_node_map).fillna(df['dip']).astype(str)

    # === Event labels ===
    event_labels = {0: "Recv", 1: "Enqu", 2: "Dequ"}

    # === Helper: group column selection ===
    base_group_cols = ['sip_short', 'dip_short', 'ProtType']
    if group_by_node:
        group_cols = ['node'] + base_group_cols
        print("🔹 Grouping by node (per-node throughput plots).")
    else:
        group_cols = base_group_cols
        print("🔹 Grouping by flow only (aggregated throughput).")

    # === Loop over events ===
    for event_val, event_name in event_labels.items():
        df_event = df[df['event'] == event_val]
        if df_event.empty:
            print(f"No data for event {event_name} ({event_val}).")
            continue

        series_list = []
        count_dict = {}

        # Group by selected columns
        for keys, group_df in df_event.groupby(group_cols):
            if group_by_node:
                node, sip_s, dip_s, prot = keys
                key_name = f"{node}_{sip_s}_to_{dip_s}_prot{str(prot).replace('/', '_')}"
            else:
                sip_s, dip_s, prot = keys
                key_name = f"{sip_s}_to_{dip_s}_prot{str(prot).replace('/', '_')}"

            # Count raw entries
            count_dict[key_name] = len(group_df)

            # Compute throughput series
            s = calculate_throughput(group_df, interval)
            if s.empty:
                continue
            series_list.append(s.rename(key_name))

        # Print raw entry counts
        print(f"\n=== Entry counts for event '{event_name}' (raw dataset rows) ===")
        for k, v in count_dict.items():
            print(f"{k}: {v}")

        if not series_list:
            print(f"No throughput series to plot for event {event_name}.")
            continue

        df_all = pd.concat(series_list, axis=1).fillna(0)

        # === Plotly Figure ===
        fig = go.Figure()

        for col in df_all.columns:
            count = count_dict.get(col, 0)
            fig.add_trace(go.Scatter(
                x=df_all.index,
                y=df_all[col],
                mode='lines',
                name=f"{col} ({count} entries)"
            ))

        grouping_text = "Per-Node" if group_by_node else "Per-Flow"
        fig.update_layout(
            title=f"{grouping_text} Throughput Over Time ({event_name})",
            xaxis_title="Time",
            yaxis_title="Throughput",
            hovermode='x unified',
            legend_title="Flow (and Node, if applicable)",
            template='plotly_white',
            height=600,
            width=1000
        )

        # === Save and show ===
        base_name = globals().get('base', 'throughput')
        out_html = f"{base_name}_{event_name}.html"
        fig.write_html(out_html)
        print(f"\n✅ Interactive plot for event '{event_name}' saved to {out_html}")
        fig.show()


In [9]:
# trace_output_file = '/home/xavid/feina/astra-sim/upc/configuration/ns3/astrasim_16nodes_ring.tr'
# trace_output_file = '/home/xavid/feina/ns-3-dev-git/three-node-trace.tr'
trace_output_file = '/home/xavid/feina/astra-sim/upc/output/comparison_run/FoldedClos/concurrent_allreduce/run_20251105_164909/ns3/astrasim_trace.tr'
df = parse_qbb_trace(trace_output_file)
interval = 0.01
plot(df, interval)

🔹 Grouping by node (per-node throughput plots).

=== Entry counts for event 'Recv' (raw dataset rows) ===
0_7_to_0_protACK: 100
1_11_to_1_protACK: 100
1_2_to_1_protUDP: 100
2_1_to_2_protACK: 100
7_0_to_7_protUDP: 100
11_1_to_11_protUDP: 100
128_0_to_7_protUDP: 100
128_1_to_11_protUDP: 100
128_1_to_2_protACK: 100
128_11_to_1_protACK: 100
128_2_to_1_protUDP: 100
128_7_to_0_protACK: 100
129_0_to_7_protUDP: 100
129_7_to_0_protACK: 100
130_1_to_11_protUDP: 100
130_11_to_1_protACK: 100
133_7_to_0_protACK: 100
135_0_to_7_protUDP: 100
135_1_to_11_protUDP: 100
135_11_to_1_protACK: 100

✅ Interactive plot for event 'Recv' saved to throughput_Recv.html



=== Entry counts for event 'Enqu' (raw dataset rows) ===
1_1_to_2_protACK: 300
7_7_to_0_protACK: 300
11_11_to_1_protACK: 300
128_0_to_7_protUDP: 100
128_1_to_11_protUDP: 100
128_1_to_2_protACK: 100
128_11_to_1_protACK: 100
128_2_to_1_protUDP: 100
128_7_to_0_protACK: 100
129_0_to_7_protUDP: 100
129_7_to_0_protACK: 100
130_1_to_11_protUDP: 100
130_11_to_1_protACK: 100
133_7_to_0_protACK: 100
135_0_to_7_protUDP: 100
135_1_to_11_protUDP: 100
135_11_to_1_protACK: 100

✅ Interactive plot for event 'Enqu' saved to throughput_Enqu.html



=== Entry counts for event 'Dequ' (raw dataset rows) ===
0_0_to_7_protUDP: 300
1_1_to_11_protUDP: 300
1_1_to_2_protACK: 300
2_2_to_1_protUDP: 300
7_7_to_0_protACK: 300
11_11_to_1_protACK: 300
128_0_to_7_protUDP: 300
128_1_to_11_protUDP: 300
128_1_to_2_protACK: 300
128_11_to_1_protACK: 300
128_2_to_1_protUDP: 300
128_7_to_0_protACK: 300
129_0_to_7_protUDP: 300
129_7_to_0_protACK: 300
130_1_to_11_protUDP: 300
130_11_to_1_protACK: 300
133_7_to_0_protACK: 300
135_0_to_7_protUDP: 300
135_1_to_11_protUDP: 300
135_11_to_1_protACK: 300

✅ Interactive plot for event 'Dequ' saved to throughput_Dequ.html
